In [1]:
import pandas as pd
import os

In [2]:
os.chdir("/Users/sushovanmandal/Documents/code/learn-gen-ai/analyticsVidhya genAI/Introduction to Deep Learning using PyTorch/Notebooks")

In [3]:
# Loading data
data = pd.read_csv('Prodigy University Dataset.csv')
# Split the data into features (X) and target (y)
data.head()

,sat_sum,hs_gpa,fy_gpa
0,508,3.40,3.18
1,488,4.00,3.33
2,464,3.75,3.25
3,380,3.75,2.42
4,428,4.00,2.63


In [4]:
# Converting data to numpy
X = data[['sat_sum', 'hs_gpa']].values
# reshape the fy_gpa into a 2D array with [data_size] rows and 1 column
y = data['fy_gpa'].values.reshape(-1, 1)
print(X.shape)
print(y.shape)

(1000, 2)
(1000, 1)


In [5]:
from sklearn.model_selection import train_test_split
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
from sklearn.preprocessing import StandardScaler

# Normalize the features so that it is easier to train the data
scaler = StandardScaler()
X_train= scaler.fit_transform(X_train)
X_test= scaler.fit_transform(X_test)

In [7]:
X_train.shape

(800, 2)

In [8]:
import torch
# Convert numpy to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

In [9]:
import torch.nn as nn

In [10]:
# Building model with 2 neurons
model = nn.Sequential(
    nn.Linear(2, 2),
    nn.Sigmoid(),
    nn.Linear(2, 1)
)

In [11]:
# Forward Propagation
preds = model(X_train_tensor)

In [12]:
preds[:5]

tensor([[-0.1953],
        [-0.2135],
        [-0.2927],
        [-0.3058],
        [-0.2247]], grad_fn=<SliceBackward0>)

In [13]:
from torch.nn import MSELoss

In [14]:
# Calculating Loss
criterion = MSELoss()
loss = criterion(preds, y_train_tensor)
print(loss)
# to learners: You may get different values

tensor(8.4005, grad_fn=<MseLossBackward0>)


# Comparing predictions on X_train with Target

In [15]:
preds[:5]

tensor([[-0.1953],
        [-0.2135],
        [-0.2927],
        [-0.3058],
        [-0.2247]], grad_fn=<SliceBackward0>)

In [16]:
y_train_tensor[:5]

tensor([[2.0000],
        [3.1100],
        [1.6300],
        [3.0200],
        [1.5500]])

In [17]:
model[0].weight

Parameter containing:
tensor([[-0.5341,  0.3355],
        [ 0.5163,  0.5503]], requires_grad=True)

In [18]:
model[2].weight

Parameter containing:
tensor([[ 0.4159, -0.3107]], requires_grad=True)

---

In [19]:
# One step of updating Weights

In [20]:
import torch.optim as optim
optimizer = optim.SGD(model.parameters(), lr = 0.001)

In [21]:
loss.backward()
optimizer.step()

In [22]:
model[0].weight

Parameter containing:
tensor([[-0.5341,  0.3356],
        [ 0.5163,  0.5503]], requires_grad=True)

In [23]:
model[2].weight

Parameter containing:
tensor([[ 0.4178, -0.3072]], requires_grad=True)

In [24]:
from torch.utils.data import TensorDataset, DataLoader

In [25]:
train_data = TensorDataset(X_train_tensor, y_train_tensor)

In [26]:
model = nn.Sequential(
    nn.Linear(2, 2),
    nn.Sigmoid(),
    nn.Linear(2, 1)
)
optimizer = optim.SGD(model.parameters(), lr = 0.001)

In [27]:
# performance on train  and test sets  before training
train_loss = criterion(model(X_train_tensor), y_train_tensor).item()
test_loss = criterion(model(X_test_tensor), y_test_tensor).item()
print(f'Without Training:\nTrain Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}')

Without Training:
Train Loss: 8.0603, Test Loss: 8.3487


In [28]:
# Looking at predictions
model(X_train_tensor)[:5]

tensor([[-0.3244],
        [-0.3258],
        [-0.3560],
        [-0.2845],
        [-0.3862]], grad_fn=<SliceBackward0>)

# Stochastic Gradient Descent

In [29]:
train_loader = DataLoader(train_data, batch_size=1, shuffle=True)
# Execute the training loop
for epoch in range(10):
    for X_batch, y_batch in train_loader:
        # Forward pass
        pred = model(X_batch)
        loss = criterion(pred, y_batch)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss = criterion(model(X_train_tensor), y_train_tensor).item()
    # print(epoch,': ', train_loss)
    test_loss = criterion(model(X_test_tensor), y_test_tensor).item()
    print(f'Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}')

Epoch 1: Train Loss: 0.4477, Test Loss: 0.5116
Epoch 2: Train Loss: 0.3975, Test Loss: 0.4473
Epoch 3: Train Loss: 0.3815, Test Loss: 0.4300
Epoch 4: Train Loss: 0.3701, Test Loss: 0.4221
Epoch 5: Train Loss: 0.3639, Test Loss: 0.4157
Epoch 6: Train Loss: 0.3600, Test Loss: 0.4130
Epoch 7: Train Loss: 0.3575, Test Loss: 0.4101
Epoch 8: Train Loss: 0.3558, Test Loss: 0.4093
Epoch 9: Train Loss: 0.3550, Test Loss: 0.4071
Epoch 10: Train Loss: 0.3545, Test Loss: 0.4061


In [30]:
# Looking at predictions
model(X_train_tensor)[:5]

tensor([[2.2895],
        [2.2841],
        [2.1093],
        [2.5526],
        [1.9307]], grad_fn=<SliceBackward0>)

# Batch Gradient Descent

In [31]:
# Reinitialising model weights
model = nn.Sequential(
    nn.Linear(2, 2),
    nn.Sigmoid(),
    nn.Linear(2, 1)
)
optimizer = optim.SGD(model.parameters(), lr = 0.001)

In [32]:
train_loader = DataLoader(train_data, batch_size=800, shuffle=True) #800 is the number of samples in train set
# Execute the training loop
for epoch in range(1000): # increasing the epochs for effective training
    for X_batch, y_batch in train_loader:
        # Forward pass
        pred = model(X_batch)
        loss = criterion(pred, y_batch)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch+1) % 100 == 0: # printing after every 100 epochs
        train_loss = criterion(model(X_train_tensor), y_train_tensor).item()
        # print(epoch,': ', train_loss)
        test_loss = criterion(model(X_test_tensor), y_test_tensor).item()
        print(f'Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}')

Epoch 100: Train Loss: 1.5278, Test Loss: 1.6685
Epoch 200: Train Loss: 1.0114, Test Loss: 1.1262
Epoch 300: Train Loss: 0.7667, Test Loss: 0.8637
Epoch 400: Train Loss: 0.6498, Test Loss: 0.7345
Epoch 500: Train Loss: 0.5924, Test Loss: 0.6687
Epoch 600: Train Loss: 0.5626, Test Loss: 0.6331
Epoch 700: Train Loss: 0.5455, Test Loss: 0.6121
Epoch 800: Train Loss: 0.5344, Test Loss: 0.5981
Epoch 900: Train Loss: 0.5260, Test Loss: 0.5878
Epoch 1000: Train Loss: 0.5191, Test Loss: 0.5795


# Mini-Batch Gradient Descent

In [33]:
# Reinitialising model weights
model = nn.Sequential(
    nn.Linear(2, 2),
    nn.Sigmoid(),
    nn.Linear(2, 1)
)
optimizer = optim.SGD(model.parameters(), lr = 0.001)

In [34]:
train_loader = DataLoader(train_data, batch_size= 64, shuffle=True) #800 is the number of samples in train set
# Execute the training loop
for epoch in range(500): # increasing the epochs for effective training
    for X_batch, y_batch in train_loader:
        # Forward pass
        pred = model(X_batch)
        loss = criterion(pred, y_batch)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch+1) % 50 == 0: # printing after every 100 epochs
        train_loss = criterion(model(X_train_tensor), y_train_tensor).item()
        # print(epoch,': ', train_loss)
        test_loss = criterion(model(X_test_tensor), y_test_tensor).item()
        print(f'Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}')

Epoch 50: Train Loss: 0.7401, Test Loss: 0.8190
Epoch 100: Train Loss: 0.6350, Test Loss: 0.6971
Epoch 150: Train Loss: 0.5733, Test Loss: 0.6297
Epoch 200: Train Loss: 0.5250, Test Loss: 0.5780
Epoch 250: Train Loss: 0.4866, Test Loss: 0.5372
Epoch 300: Train Loss: 0.4562, Test Loss: 0.5052
Epoch 350: Train Loss: 0.4323, Test Loss: 0.4804
Epoch 400: Train Loss: 0.4136, Test Loss: 0.4612
Epoch 450: Train Loss: 0.3990, Test Loss: 0.4464
Epoch 500: Train Loss: 0.3877, Test Loss: 0.4351


Run all cells till here

Let's quickly run the model using the new techniques we just looked at that is GD with Momentum and Nesterov Momentum. Let's begin with GD with momentum. 

# Gradient Descent with Momentum

In [35]:
# Reinitialising model weights
model = nn.Sequential(
    nn.Linear(2, 2),
    nn.Sigmoid(),
    nn.Linear(2, 1)
)
optimizer = optim.SGD(model.parameters(), lr = 0.001, momentum=0.9)

Here, we've introduced a momentum of 0.9 to the SGD optimizer. Ensure that you add the momentum parameter to the optimizer else the model would simply use the basic SGD.

In [36]:
train_loader = DataLoader(train_data, batch_size= 64, shuffle=True) #800 is the number of samples in train set
# Execute the training loop
for epoch in range(500): # increasing the epochs for effective training
    for X_batch, y_batch in train_loader:
        # Forward pass
        pred = model(X_batch)
        loss = criterion(pred, y_batch)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch+1) % 50 == 0: # printing after every 100 epochs
        train_loss = criterion(model(X_train_tensor), y_train_tensor).item()
        # print(epoch,': ', train_loss)
        test_loss = criterion(model(X_test_tensor), y_test_tensor).item()
        print(f'Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}')

Epoch 50: Train Loss: 0.3442, Test Loss: 0.4017
Epoch 100: Train Loss: 0.3423, Test Loss: 0.4010
Epoch 150: Train Loss: 0.3420, Test Loss: 0.4011
Epoch 200: Train Loss: 0.3418, Test Loss: 0.4007
Epoch 250: Train Loss: 0.3415, Test Loss: 0.4007
Epoch 300: Train Loss: 0.3414, Test Loss: 0.4002
Epoch 350: Train Loss: 0.3412, Test Loss: 0.4005
Epoch 400: Train Loss: 0.3411, Test Loss: 0.3998
Epoch 450: Train Loss: 0.3409, Test Loss: 0.3998
Epoch 500: Train Loss: 0.3408, Test Loss: 0.4004


Here, we just observed how with a momentum 0.9 we reached to lower value of loss in a much faster manner! 

Let's quickly run the Nesterov Momentum on our dataset and evaluate the losses.

# Nesterov Momentum

The code is the almost the same as SGD with Momentum, but all you have to do is set the nesterov parameter to true.

In [37]:
# Reinitialising model weights
model = nn.Sequential(
    nn.Linear(2, 2),
    nn.Sigmoid(),
    nn.Linear(2, 1)
)
optimizer = optim.SGD(model.parameters(), lr = 0.001, momentum=0.9, nesterov=True)

In [38]:
train_loader = DataLoader(train_data, batch_size= 64, shuffle=True) #800 is the number of samples in train set
# Execute the training loop
for epoch in range(500): # increasing the epochs for effective training
    for X_batch, y_batch in train_loader:
        # Forward pass
        pred = model(X_batch)
        loss = criterion(pred, y_batch)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch+1) % 50 == 0: # printing after every 100 epochs
        train_loss = criterion(model(X_train_tensor), y_train_tensor).item()
        # print(epoch,': ', train_loss)
        test_loss = criterion(model(X_test_tensor), y_test_tensor).item()
        print(f'Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}')

Epoch 50: Train Loss: 0.3644, Test Loss: 0.4176
Epoch 100: Train Loss: 0.3471, Test Loss: 0.4035
Epoch 150: Train Loss: 0.3453, Test Loss: 0.4026
Epoch 200: Train Loss: 0.3444, Test Loss: 0.4025
Epoch 250: Train Loss: 0.3438, Test Loss: 0.4024
Epoch 300: Train Loss: 0.3432, Test Loss: 0.4021
Epoch 350: Train Loss: 0.3428, Test Loss: 0.4013
Epoch 400: Train Loss: 0.3424, Test Loss: 0.4013
Epoch 450: Train Loss: 0.3421, Test Loss: 0.4010
Epoch 500: Train Loss: 0.3418, Test Loss: 0.4009


The loss calculations are almost at par with the GD with momentum. Our final train and test loss stand at VALUE and VALUE.

Things are getting interesting aren’t they. Feel free to revisit the concepts we have covered so far before we move to the next optimizer which is AdaGrad..


# AdaGrad

Next, let's quickly run each of the different optimizers we just looked at starting with AdaGrad. Note that here we've to specify optim.Adagead(model.parameters()) to intialize the model with Adagrad. If you recall, Adagrad adjusts the learning rates of each parameter based on the historical gradients. 

In [39]:
# Reinitialising model weights
model = nn.Sequential(
    nn.Linear(2, 2),
    nn.Sigmoid(),
    nn.Linear(2, 1)
)
optimizer = optim.Adagrad(model.parameters())

In [40]:
train_loader = DataLoader(train_data, batch_size= 64, shuffle=True) #800 is the number of samples in train set
# Execute the training loop
for epoch in range(500): # increasing the epochs for effective training
    for X_batch, y_batch in train_loader:
        # Forward pass
        pred = model(X_batch)
        loss = criterion(pred, y_batch)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch+1) % 50 == 0: # printing after every 100 epochs
        train_loss = criterion(model(X_train_tensor), y_train_tensor).item()
        # print(epoch,': ', train_loss)
        test_loss = criterion(model(X_test_tensor), y_test_tensor).item()
        print(f'Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}')

Epoch 50: Train Loss: 1.7559, Test Loss: 1.9081
Epoch 100: Train Loss: 0.9432, Test Loss: 1.0619
Epoch 150: Train Loss: 0.6139, Test Loss: 0.7119
Epoch 200: Train Loss: 0.4699, Test Loss: 0.5539
Epoch 250: Train Loss: 0.4048, Test Loss: 0.4793
Epoch 300: Train Loss: 0.3751, Test Loss: 0.4433
Epoch 350: Train Loss: 0.3615, Test Loss: 0.4254
Epoch 400: Train Loss: 0.3552, Test Loss: 0.4163
Epoch 450: Train Loss: 0.3522, Test Loss: 0.4114
Epoch 500: Train Loss: 0.3507, Test Loss: 0.4087


Adagrad has given us a high intial loss but the final loss values of VALUE on the train data and VALUE on the test data. Next let's try RMSProp!

# RMS Prop

Just like we did with Adagrad earlier, here we need to use optim.RMSprop to intitialize the model with RMSProp. Even though we are going to use the default parameters, RMSprop also has parameters such as, learning rate, momentum etc, which you can feel free to try out! Let's run the code.

In [41]:
# Reinitialising model weights
model = nn.Sequential(
    nn.Linear(2, 2),
    nn.Sigmoid(),
    nn.Linear(2, 1)
)
optimizer = optim.RMSprop(model.parameters())

In [42]:
train_loader = DataLoader(train_data, batch_size= 64, shuffle=True) #800 is the number of samples in train set
# Execute the training loop
for epoch in range(500): # increasing the epochs for effective training
    for X_batch, y_batch in train_loader:
        # Forward pass
        pred = model(X_batch)
        loss = criterion(pred, y_batch)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch+1) % 50 == 0: # printing after every 100 epochs
        train_loss = criterion(model(X_train_tensor), y_train_tensor).item()
        # print(epoch,': ', train_loss)
        test_loss = criterion(model(X_test_tensor), y_test_tensor).item()
        print(f'Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}')

Epoch 50: Train Loss: 0.3427, Test Loss: 0.3997
Epoch 100: Train Loss: 0.3401, Test Loss: 0.4030
Epoch 150: Train Loss: 0.3385, Test Loss: 0.3997
Epoch 200: Train Loss: 0.3386, Test Loss: 0.3996
Epoch 250: Train Loss: 0.3385, Test Loss: 0.3987
Epoch 300: Train Loss: 0.3402, Test Loss: 0.4060
Epoch 350: Train Loss: 0.3387, Test Loss: 0.4037
Epoch 400: Train Loss: 0.3376, Test Loss: 0.4011
Epoch 450: Train Loss: 0.3377, Test Loss: 0.3980
Epoch 500: Train Loss: 0.3370, Test Loss: 0.4000


With RMSProp we have clearly achieved our lowest loss values so far. We have got final loss values of VALUE on the train data and VALUE on the test data

Next, let's try the Adam Optimizer. For this, we need to use optim.Adam. Take a look at the code.

# Adam

In [43]:
# Reinitialising model weights
model = nn.Sequential(
    nn.Linear(2, 2),
    nn.Sigmoid(),
    nn.Linear(2, 1)
)
optimizer = optim.Adam(model.parameters())

In [44]:
train_loader = DataLoader(train_data, batch_size= 64, shuffle=True) #800 is the number of samples in train set
# Execute the training loop
for epoch in range(500): # increasing the epochs for effective training
    for X_batch, y_batch in train_loader:
        # Forward pass
        pred = model(X_batch)
        loss = criterion(pred, y_batch)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch+1) % 50 == 0: # printing after every 100 epochs
        train_loss = criterion(model(X_train_tensor), y_train_tensor).item()
        # print(epoch,': ', train_loss)
        test_loss = criterion(model(X_test_tensor), y_test_tensor).item()
        print(f'Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}')

Epoch 50: Train Loss: 1.7711, Test Loss: 1.9211
Epoch 100: Train Loss: 0.5004, Test Loss: 0.5784
Epoch 150: Train Loss: 0.3688, Test Loss: 0.4227
Epoch 200: Train Loss: 0.3610, Test Loss: 0.4130
Epoch 250: Train Loss: 0.3579, Test Loss: 0.4111
Epoch 300: Train Loss: 0.3547, Test Loss: 0.4089
Epoch 350: Train Loss: 0.3516, Test Loss: 0.4072
Epoch 400: Train Loss: 0.3488, Test Loss: 0.4056
Epoch 450: Train Loss: 0.3466, Test Loss: 0.4036
Epoch 500: Train Loss: 0.3449, Test Loss: 0.4034


The Adam optimizer too has given us good overall performance. As per our observations, RMSProp is the best of the lot for optimizing the loss in our case. You've now explored a variety of optimization algorithms, each with unique approaches to navigating the complex landscape of neural network training. With this solid foundation of concepts, I’m sure you're well-equipped to apply them in practice. By thoughtfully selecting and implementing the right optimizer, our goal is to fine-tune our model's learning process for better performance and results. I’ll see you in the next video in which we shall build advanced neural models for real world projects.